In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HF_NEW")
assert hf_token, "Set HF_TOKEN (or HF_NEW) in your .env file at the repo root"


In [ ]:
from datasets import load_dataset
LOAD_SPECIFIC_FILE = True
dataset_name = "1024m/LID"
if LOAD_SPECIFIC_FILE:
    file_path = "Data_Hackathon/LID-20000.parquet"
    dataset = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=hf_token)["train"]
else:
    dataset = load_dataset(dataset_name, token=hf_token)["train"]
print(f"dataset  : {dataset_name}")
print(f"samples  : {len(dataset)}")
print(f"columns  : {dataset.column_names}")
print(f"size     : {dataset.dataset_size / 1024**2:.3f} MB")

In [ ]:
df_data = dataset.to_pandas()
df_data

In [ ]:
# `unicodeblock` is included in the base `uv sync` dependency set. From
# the repo root, run:
#     uv sync
# Then launch: jupyter lab


In [ ]:
from unicodeblock import blocks
blocks.of('א')  # returns 'HEBREW'

In [ ]:
from unicodeblock import blocks
from collections import Counter
import numpy as np
from tqdm import tqdm
BATCH_SIZE = 10000
texts = df_data['text'].values
results = np.empty(len(texts), dtype=object)
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    for j, text in enumerate(batch):
        results[i+j] = Counter(blocks.of(c) for c in str(text) if c.strip())
df_data['block_dist'] = results

In [ ]:
from huggingface_hub import login
from datasets import load_dataset
from unicodeblock import blocks
from collections import Counter
from tqdm import tqdm
import numpy as np

# hf_token is loaded from .env in cell 0.
login(token=hf_token)
ds = load_dataset("commoncrawl/CommonLID", split="test")
df_commonLID = ds.to_pandas()
texts = df_commonLID['text'].values
block_dists = np.empty(len(texts), dtype=object)
for i in tqdm(range(0, len(texts), 10000)):
    batch = texts[i:i+10000]
    for j, text in enumerate(batch):
        block_dists[i+j] = Counter(blocks.of(c) for c in str(text) if c.strip())
df_commonLID['block_dist'] = block_dists


In [ ]:
df_data

In [ ]:
for code in df_data['ISO-693-3'].unique():
    exec(f"df_{code} = df_data[df_data['ISO-693-3'] == '{code}'].reset_index(drop=True)")

In [ ]:
for code in df_data['ISO-693-3'].unique():
    exec(f"""
df_{code}['COMM'] = df_{code}['block_dist'].apply(lambda d: max(d, key=d.get) if d else None)
df_{code}['THRESH'] = df_{code}.apply(lambda r: float(f"{{r['block_dist'].get(r['COMM'], 0) / sum(r['block_dist'].values()):.2f}}") if r['block_dist'] else None, axis=1)
""")

In [ ]:
all_blocks = set()
for d in df_data['block_dist']:
    if d is not None:
        all_blocks.update(k for k in d.keys() if k is not None)
print(sorted(all_blocks))

In [ ]:
print(len(all_blocks))

## **noise blocks**

In [ ]:
noise_blocks = [
    'DIGIT','BASIC_PUNCTUATION','GENERAL_PUNCTUATION','CURRENCY_SYMBOLS',
    'SUPERSCRIPTS_AND_SUBSCRIPTS','NUMBER_FORMS','MATHEMATICAL_OPERATORS',
    'MISCELLANEOUS_MATHEMATICAL_SYMBOLS_A','ARROWS','BOX_DRAWING',
    'CONTROL_PICTURES','DINGBATS','GEOMETRIC_SHAPES','MISCELLANEOUS_SYMBOLS',
    'MISCELLANEOUS_SYMBOLS_AND_ARROWS','MISCELLANEOUS_SYMBOLS_AND_PICTOGRAPHS',
    'SMALL_FORM_VARIANTS','SPECIALS','PRIVATE_USE_AREA','VARIATION_SELECTORS',
    'VARIATION_SELECTORS_SUPPLEMENT','ENCLOSED_ALPHANUMERICS',
    'ENCLOSED_CJK_LETTERS_AND_MONTHS','LETTERLIKE_SYMBOLS','FULLWIDTH_DIGIT',
    'HALFWIDTH_AND_FULLWIDTH_FORMS','CUNEIFORM','OLD_PERSIAN',
    'OLD_SOUTH_ARABIAN','PHOENICIAN', 'ALCHEMICAL_SYMBOLS','ANCIENT_GREEK_NUMBERS',
    'BLOCK_ELEMENTS','BRAILLE_PATTERNS','EMOTICONS','ANCIENT_SYMBOLS',
    'MATHEMATICAL_ALPHANUMERIC_SYMBOLS','MISCELLANEOUS_MATHEMATICAL_SYMBOLS_B',
    'MISCELLANEOUS_TECHNICAL','MUSICAL_SYMBOLS','PLAYING_CARDS',
    'SUPPLEMENTAL_ARROWS_A','SUPPLEMENTAL_ARROWS_B',
    'SUPPLEMENTAL_MATHEMATICAL_OPERATORS','SUPPLEMENTARY_PRIVATE_USE_AREA_A',
    'SUPPLEMENTARY_PRIVATE_USE_AREA_B','TAGS','TRANSPORT_AND_MAP_SYMBOLS',
    'YIJING_HEXAGRAM_SYMBOLS', 'ENCLOSED_ALPHANUMERIC_SUPPLEMENT'
]
rows = []
for code in df_data['ISO-693-3'].unique():
    subset = df_data[df_data['ISO-693-3'] == code]['block_dist']
    total_chars = subset.apply(lambda d: sum(d.values()) if d else 0).sum()
    row = {'ISO-693-3': code}
    for block in noise_blocks:
        block_count = subset.apply(lambda d: d.get(block, 0) if d else 0).sum()
        row[block] = float(f"{block_count / total_chars:.3f}") if total_chars > 0 else 0.0
    rows.append(row)

In [ ]:
import pandas as pd
df_rem = pd.DataFrame(rows).set_index('ISO-693-3')
df_rem

In [ ]:
import matplotlib.pyplot as plt
y_max = df_rem.max().max()
for col in df_rem.columns:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(df_rem.index, df_rem[col])
    ax.set_ylim(0, y_max)
    ax.set_title(col)
    ax.set_xlabel('ISO-693-3')
    ax.set_ylabel('distribution')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

half width and full width block can be used to pre-classify chinese and japanese

In [ ]:
for code in df_data['ISO-693-3'].unique():
    subset = df_data[df_data['ISO-693-3'] == code]['block_dist']
    count = subset.apply(lambda d: 1 if d and d.get('HALFWIDTH_AND_FULLWIDTH_FORMS', 0) > 0 else 0).sum()
    if count > 0:
        print(f"{code} : {count}")

In [ ]:
cjk_blocks = ['CJK_UNIFIED_IDEOGRAPHS', 'CJK_UNIFIED_IDEOGRAPHS_EXTENSION_A', 'CJK_UNIFIED_IDEOGRAPHS_EXTENSION_B', 'CJK_UNIFIED_IDEOGRAPHS_EXTENSION_C', 'CJK_UNIFIED_IDEOGRAPHS_EXTENSION_D']
jpn_signal = lambda d: d.get('HIRAGANA',0) + d.get('KATAKANA',0) + d.get('HALFWIDTH_AND_FULLWIDTH_FORMS',0) > 0
zho_signal = lambda d: any(d.get(b,0) > 0 for b in cjk_blocks)
print("jpn caught  :", df_data[df_data['ISO-693-3']=='jpn']['block_dist'].apply(jpn_signal).sum())
print("zho caught  :", df_data[df_data['ISO-693-3']=='zho']['block_dist'].apply(zho_signal).sum())
print("unnecessary :", df_data[~df_data['ISO-693-3'].isin(['jpn','zho'])]['block_dist'].apply(lambda d: jpn_signal(d) or zho_signal(d)).sum())

In [ ]:
jpn_signal = lambda d: d.get('HIRAGANA',0) + d.get('KATAKANA',0) > 0
kor_signal = lambda d: d.get('HANGUL_SYLLABLES',0) + d.get('HANGUL_JAMO',0) + d.get('HANGUL_COMPATIBILITY_JAMO',0) > 0
zho_signal = lambda d: not jpn_signal(d) and not kor_signal(d) and any(d.get(b,0) > 0 for b in cjk_blocks)
def classify_cjk_v2(d):
    if not d:
        return None
    has_hira_kata = d.get('HIRAGANA',0) + d.get('KATAKANA',0) > 0
    has_fullwidth = d.get('HALFWIDTH_AND_FULLWIDTH_FORMS',0) > 0
    has_cjk = any(d.get(b,0) > 0 for b in cjk_blocks)
    if has_hira_kata:
        return 'jpn'
    if has_fullwidth and not has_cjk:
        return 'jpn'
    if has_cjk:
        return 'zho'
    return None
df_commonLID['cjk_flag'] = df_commonLID['block_dist'].apply(classify_cjk_v2)
jpn_rows = df_commonLID[df_commonLID['tag'] == 'jpn']
zho_rows = df_commonLID[df_commonLID['tag'] == 'zho']
rest_rows = df_commonLID[~df_commonLID['tag'].isin(['jpn','zho'])]
jpn_correct = (jpn_rows['cjk_flag'] == 'jpn').sum()
jpn_incorrect = len(jpn_rows) - jpn_correct
zho_correct = (zho_rows['cjk_flag'] == 'zho').sum()
zho_incorrect = len(zho_rows) - zho_correct
rest_correct = rest_rows['cjk_flag'].isna().sum()
rest_incorrect = len(rest_rows) - rest_correct
print(f"jpn  : acc {jpn_correct/len(jpn_rows):.2f} | correct {jpn_correct} | incorrect {jpn_incorrect}")
print(f"zho  : acc {zho_correct/len(zho_rows):.2f} | correct {zho_correct} | incorrect {zho_incorrect}")
print(f"rest : acc {rest_correct/len(rest_rows):.2f} | correct {rest_correct} | incorrect {rest_incorrect}")

In [ ]:
df_data['cjk_flag'] = df_data['block_dist'].apply(classify_cjk_v2)
jpn_rows = df_data[df_data['ISO-693-3'] == 'jpn']
zho_rows = df_data[df_data['ISO-693-3'] == 'zho']
rest_rows = df_data[~df_data['ISO-693-3'].isin(['jpn','zho'])]
jpn_correct = (jpn_rows['cjk_flag'] == 'jpn').sum()
jpn_incorrect = len(jpn_rows) - jpn_correct
zho_correct = (zho_rows['cjk_flag'] == 'zho').sum()
zho_incorrect = len(zho_rows) - zho_correct
rest_correct = rest_rows['cjk_flag'].isna().sum()
rest_incorrect = len(rest_rows) - rest_correct
print(f"jpn  : acc {jpn_correct/len(jpn_rows):.2f} | correct {jpn_correct} | incorrect {jpn_incorrect}")
print(f"zho  : acc {zho_correct/len(zho_rows):.2f} | correct {zho_correct} | incorrect {zho_incorrect}")
print(f"rest : acc {rest_correct/len(rest_rows):.2f} | correct {rest_correct} | incorrect {rest_incorrect}")

### **Rest**

In [ ]:
for code in df_data['ISO-693-3'].unique():
    exec(f"""
lang_val = df_{code}['lang'].iloc[0]
comm_vals = df_{code}['COMM'].value_counts().index.tolist()
thresh_val = f"{{df_{code}['THRESH'].min():.2f}}" if len(comm_vals) < 2 else "NA"
if len(comm_vals) > 3:
    comm_str = "['" + "', '".join(comm_vals[:3]) + "', ..."
else:
    comm_str = str(comm_vals)
print(f"df_{code} : {{lang_val}} : {{comm_str}} : {{thresh_val}}")
""")

In [ ]:
afr_langs = ['amh', 'hau', 'ibo', 'mlg', 'sna', 'swh', 'wol', 'xho', 'yor', 'zul']
sea_langs = ['tgl', 'msa', 'ind', 'vie', 'jav', 'khm', 'tha', 'lao', 'zho', 'mya', 'jpn', 'kor']
ind_langs = ['hin', 'mar', 'ben', 'guj', 'pan', 'tam', 'tel', 'nep']
mea_langs = ['ara', 'fas', 'urd', 'tur', 'mlt', 'heb']
eng_langs = ['eng']
eur_langs = ['nld', 'fra', 'ita', 'por', 'ron', 'spa', 'ces', 'pol', 'ukr', 'rus',
             'ell', 'deu', 'dan', 'swe', 'nor', 'cat', 'glg', 'cym', 'gle', 'eus',
             'hrv', 'lav', 'lit', 'slk', 'slv', 'est', 'fin', 'hun', 'srp', 'bul']

In [ ]:
unique_script  = ['amh','mya','tam','tel','ben','guj','pan','khm','heb','jpn','kor','zho','tha','lao']
devanagari     = ['hin','mar','nep']
arabic_script  = ['ara','fas','urd']
cyrillic       = ['rus','ukr','bul','srp']
greek          = ['ell']
latin          = ['wol','swh','swe','fin','slk','dan','hun','ces','xho','gle','tur','lav','eng','glg','ita','por','pol','hau','nld','ron','spa','tgl','slv','cat','fra','cym','mlg','ind','vie','eus','yor','est','jav','sna','lit','ibo','mlt','zul','nor','msa','deu','hrv']
import re
script_patterns = {
    'amh': re.compile(r'[\u1200-\u137F]'),
    'mya': re.compile(r'[\u1000-\u109F]'),
    'tam': re.compile(r'[\u0B80-\u0BFF]'),
    'tel': re.compile(r'[\u0C00-\u0C7F]'),
    'ben': re.compile(r'[\u0980-\u09FF]'),
    'guj': re.compile(r'[\u0A80-\u0AFF]'),
    'pan': re.compile(r'[\u0A00-\u0A7F]'),
    'khm': re.compile(r'[\u1780-\u17FF]'),
    'heb': re.compile(r'[\u0590-\u05FF]'),
    'jpn': re.compile(r'[\u3040-\u309F\u30A0-\u30FF]'),
    'kor': re.compile(r'[\uAC00-\uD7AF\u1100-\u11FF]'),
    'zho': re.compile(r'[\u4E00-\u9FFF\u3400-\u4DBF]'),
    'tha': re.compile(r'[\u0E00-\u0E7F]'),
    'lao': re.compile(r'[\u0E80-\u0EFF]'),
    'hin': re.compile(r'[\u0900-\u097F]'),
    'mar': re.compile(r'[\u0900-\u097F]'),
    'nep': re.compile(r'[\u0900-\u097F]'),
    'ara': re.compile(r'[\u0600-\u06FF]'),
    'fas': re.compile(r'[\u0600-\u06FF]'),
    'urd': re.compile(r'[\u0600-\u06FF]'),
    'wol': re.compile(r'[\u0600-\u06FF]'),
    'rus': re.compile(r'[\u0400-\u04FF]'),
    'ukr': re.compile(r'[\u0400-\u04FF]'),
    'bul': re.compile(r'[\u0400-\u04FF]'),
    'srp': re.compile(r'[\u0400-\u04FF]'),
    'ell': re.compile(r'[\u0370-\u03FF\u1F00-\u1FFF]'),
}

In [ ]:
group_map = {}
for code in unique_script: group_map[code] = 'unique_script'
for code in devanagari:    group_map[code] = 'devanagari'
for code in arabic_script: group_map[code] = 'arabic_script'
for code in cyrillic:      group_map[code] = 'cyrillic'
for code in greek:         group_map[code] = 'greek'
for code in latin:         group_map[code] = 'latin'
def classify_regex(text):
    text = str(text)
    scores = {code: len(pattern.findall(text)) for code, pattern in script_patterns.items()}
    best = max(scores, key=scores.get)
    if scores[best] == 0:
        return 'latin'
    return best
df_data['regex_pred'] = df_data['text'].apply(classify_regex)
df_data['regex_group'] = df_data['regex_pred'].map(group_map).fillna('latin')
df_data['true_group'] = df_data['ISO-693-3'].map(group_map)
groups = {
    'unique_script': unique_script,
    'devanagari':    devanagari,
    'arabic_script': arabic_script,
    'cyrillic':      cyrillic,
    'greek':         greek,
    'latin':         latin,
}
for grp_name, codes in groups.items():
    subset = df_data[df_data['ISO-693-3'].isin(codes)]
    correct = (subset['regex_group'] == grp_name).sum()
    acc = correct / len(subset)
    print(f"{grp_name:<20} : acc {acc:.2f} | correct {correct} | incorrect {len(subset)-correct}")

In [ ]:
df_misc_1 = df_data[df_data['ISO-693-3'].isin(unique_script)  & (df_data['regex_group'] != 'unique_script')].reset_index(drop=True)
df_misc_2 = df_data[df_data['ISO-693-3'].isin(devanagari)     & (df_data['regex_group'] != 'devanagari')].reset_index(drop=True)
df_misc_3 = df_data[df_data['ISO-693-3'].isin(arabic_script)  & (df_data['regex_group'] != 'arabic_script')].reset_index(drop=True)
df_misc_4 = df_data[df_data['ISO-693-3'].isin(cyrillic)       & (df_data['regex_group'] != 'cyrillic')].reset_index(drop=True)
df_misc_5 = df_data[df_data['ISO-693-3'].isin(greek)          & (df_data['regex_group'] != 'greek')].reset_index(drop=True)
df_misc_6 = df_data[df_data['ISO-693-3'].isin(latin)          & (df_data['regex_group'] != 'latin')].reset_index(drop=True)

In [ ]:
subset = df_data[df_data['ISO-693-3'].isin(unique_script)]
for code in unique_script:
    lang_rows = subset[subset['ISO-693-3'] == code]
    correct = (lang_rows['regex_pred'] == code).sum()
    acc = correct / len(lang_rows)
    print(f"{code} : acc {acc:.2f} | correct {correct} | incorrect {len(lang_rows)-correct}")

In [ ]:
jpn_rows = df_data[df_data['ISO-693-3'] == 'jpn']
print(jpn_rows[jpn_rows['regex_pred'] != 'jpn']['regex_pred'].value_counts())

In [ ]:
mask = (df_data['regex_pred'] == 'zho') & (df_data['text'].apply(lambda t: bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF]', str(t)))))
df_data.loc[mask, 'regex_pred'] = 'jpn'
print(mask.sum(), "rows patched")

In [ ]:
df_data['regex_group'] = df_data['regex_pred'].map(group_map).fillna('latin')
for grp_name, codes in groups.items():
    subset = df_data[df_data['ISO-693-3'].isin(codes)]
    correct = (subset['regex_group'] == grp_name).sum()
    acc = correct / len(subset)
    print(f"{grp_name:<20} : acc {acc:.2f} | correct {correct} | incorrect {len(subset)-correct}")
subset = df_data[df_data['ISO-693-3'].isin(unique_script)]
for code in unique_script:
    lang_rows = subset[subset['ISO-693-3'] == code]
    correct = (lang_rows['regex_pred'] == code).sum()
    acc = correct / len(lang_rows)
    print(f"{code} : acc {acc:.2f} | correct {correct} | incorrect {len(lang_rows)-correct}")

In [ ]:
print(df_data[(df_data['ISO-693-3'] != 'jpn') & (df_data['ISO-693-3'] != 'zho') & (df_data['regex_pred'] == 'jpn')]['ISO-693-3'].value_counts())

In [ ]:
df_cached = df_data[(df_data['ISO-693-3'].isin(['kor','vie'])) & (df_data['regex_pred'] == 'jpn')].copy()
df_cached

In [ ]:
def is_latin_dominant(text):
    text = str(text)
    latin_count = len(re.findall(r'[\u0041-\u007A\u00C0-\u024F]', text))
    total = len([c for c in text if c.strip()])
    return total > 0 and latin_count / total >= 0.5
mask = (~df_data['ISO-693-3'].isin(['jpn','zho'])) & (df_data['regex_pred'] != df_data['true_group'].map(lambda x: x if x != 'latin' else 'latin')) & df_data['text'].apply(is_latin_dominant)
df_data.loc[mask, 'regex_pred'] = 'latin'
print(mask.sum(), "rows patched")

In [ ]:
df_data['regex_group'] = df_data['regex_pred'].map(group_map).fillna('latin')

In [ ]:
for grp_name, codes in groups.items():
    subset = df_data[df_data['ISO-693-3'].isin(codes)]
    correct = (subset['regex_group'] == grp_name).sum()
    acc = correct / len(subset)
    print(f"{grp_name:<20} : acc {acc:.2f} | correct {correct} | incorrect {len(subset)-correct}")
print("###################")
subset = df_data[df_data['ISO-693-3'].isin(unique_script)]
for code in unique_script:
    lang_rows = subset[subset['ISO-693-3'] == code]
    correct = (lang_rows['regex_pred'] == code).sum()
    acc = correct / len(lang_rows)
    print(f"{code} : acc {acc:.2f} | correct {correct} | incorrect {len(lang_rows)-correct}")

In [ ]:
latin          = ['wol','swh','swe','fin','slk','dan','hun','ces','xho','gle','tur','lav','eng','glg','ita','por','pol','hau','nld','ron','spa','tgl','slv','cat','fra','cym','mlg','ind','vie','eus','yor','est','jav','sna','lit','ibo','mlt','zul','nor','msa','deu','hrv']

In [ ]:
df_misc_3

# **Issues in dataset itself - the newspapers we used marked all _lines_ of a {LANG} newspaper as a {LANG} newspaper**

In [ ]:
df_misc_1

# **Issues in benchmarks as well - texts which are completely alphanumeric in english are marked as chinese for example in CommonLID**

In [ ]:
df_commonLID[df_commonLID['tag'] == 'zho'][df_commonLID[df_commonLID['tag'] == 'zho']['text'].apply(lambda t: all(ord(c) < 128 for c in str(t)))].head(20)